# Entropy-Delta baseline — D4C, Wang, PRISM, RepairLLaMA, SRepair

InCoder-6.7B, prefix+suffix infilling, 2048-token window (Yang et al., ICSE 2025).
`entropy_delta = E(buggy region) - E(patched region)` — positive = more natural = more likely correct.

**Runtime → Change runtime type → L4 GPU.**

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
CONTENT = '/content'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install transformers accelerate scikit-learn pandas

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## Inputs — upload all 10 files

In [ ]:
need = [f for f in [
    'wang_patches_export.csv', 'wang_prms_results.csv',
    'prism_patches_export.csv', 'prism_prms_results.csv',
    'repairllama_patches_export.csv', 'repairllama_prms_results.csv',
    'srepair_patches_export.csv', 'srepair_prms_results.csv',
    'pred_d4c_gpt_archived.csv', 'd4c_prms_results.csv',
] if not os.path.exists(f'{CONTENT}/{f}')]
if need:
    from google.colab import files
    print('Upload:', need)
    for name, data in files.upload().items():
        open(f'{CONTENT}/{name}', 'wb').write(data)
print('All files present:', all(os.path.exists(f'{CONTENT}/{f}') for f in [
    'wang_patches_export.csv','prism_patches_export.csv',
    'repairllama_patches_export.csv','srepair_patches_export.csv', 'pred_d4c_gpt_archived.csv']))

Upload: ['wang_patches_export.csv', 'wang_prms_results.csv', 'prism_patches_export.csv', 'prism_prms_results.csv', 'repairllama_patches_export.csv', 'repairllama_prms_results.csv', 'srepair_patches_export.csv', 'srepair_prms_results.csv', 'pred_d4c_gpt_archived.csv', 'd4c_prms_results.csv']


Saving d4c_prms_results.csv to d4c_prms_results.csv
Saving pred_d4c_gpt_archived.csv to pred_d4c_gpt_archived.csv
Saving prism_patches_export.csv to prism_patches_export.csv
Saving prism_prms_results.csv to prism_prms_results.csv
Saving repairllama_patches_export.csv to repairllama_patches_export.csv
Saving repairllama_prms_results.csv to repairllama_prms_results.csv
Saving srepair_patches_export.csv to srepair_patches_export.csv
Saving srepair_prms_results.csv to srepair_prms_results.csv
Saving wang_patches_export.csv to wang_patches_export.csv
Saving wang_prms_results.csv to wang_prms_results.csv
All files present: True


In [ ]:
prems = pd.read_csv(f'{BASE}/d4c_prms_results.csv')
print(len(prems), prems.slug.duplicated().sum())
prems_dedup = prems.drop_duplicates(subset='slug', keep='first')
print(len(prems_dedup))
prems_dedup.to_csv(f'{BASE}/d4c_prms_results.csv', index=False)

112 13
99


## Optional: save to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/entropy-delta'
os.makedirs(OUT_DIR, exist_ok=True)
print('Results will be saved to', OUT_DIR)

Mounted at /content/drive
Results will be saved to /content/drive/MyDrive/entropy-delta


## Code

In [ ]:
#!/usr/bin/env python3

import argparse, csv, os, sys
import numpy as np
import pandas as pd

BASE = CONTENT

DATASETS = {
    'd4c':         (None,                             'd4c_prms_results.csv'),
    'wang':        ('wang_patches_export.csv',        'wang_prms_results.csv'),
    'prism':       ('prism_patches_export.csv',       'prism_prms_results.csv'),
    'repairllama': ('repairllama_patches_export.csv', 'repairllama_prms_results.csv'),
    'srepair':     ('srepair_patches_export.csv',     'srepair_prms_results.csv'),
}

D4C_PRED_CSV = 'pred_d4c_gpt_archived.csv'

MAX_TOKENS = 2048
HEADER     = '<| file ext=.java |>\n'
MASK       = '<|mask:0|>'
EOM        = '<|endofmask|>'
EMPTY      = ''

FIELDS = ['dataset', 'slug', 'label', 'assessable', 'prmsc',
          'e_orig', 'e_patch', 'entropy_delta', 'token_budget', 'status']


def join(lines):
    return '\n'.join(lines) + '\n' if lines else EMPTY

def patch_region(bug_code, patch_code):
    import difflib, textwrap
    def norm(c): return textwrap.dedent(str(c).replace('\r\n','\n').replace('\r','\n').strip('\n'))
    b = norm(bug_code).split('\n')
    p = norm(patch_code).split('\n')
    sm = difflib.SequenceMatcher(None, [l.strip() for l in b], [l.strip() for l in p], autojunk=False)
    ops = [op for op in sm.get_opcodes() if op[0] != 'equal']
    if not ops:
        return None
    i1, j1 = ops[0][1], ops[0][3]
    i2, j2 = ops[-1][2], ops[-1][4]
    return b[:i1], b[i1:i2], p[j1:j2], b[i2:]


class Scorer:
    def __init__(self, model_name):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch = torch
        self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        kw = dict(low_cpu_mem_usage=True)
        if self.dev == 'cuda':
            kw['torch_dtype'] = torch.float16
            if model_name == 'facebook/incoder-6B':
                kw['revision'] = 'float16'
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.tok.model_max_length = 10**6
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, attn_implementation='sdpa', **kw)
        except Exception:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **kw)
        self.model = self.model.to(self.dev).eval()
        self.eom = self.ids(EOM)

    def ids(self, text):
        return self.tok(text, add_special_tokens=False)['input_ids']

    def _fit(self, prefix_lines, suffix_lines, n_mid, budget):
        pre, suf = list(prefix_lines), list(suffix_lines)
        dropped = 0
        bos = [self.tok.bos_token_id] if self.tok.bos_token_id is not None else []
        while True:
            ctx = HEADER + join(pre)
            ctx_ids = bos + self.ids(ctx + MASK + '\n'.join(suf) + MASK)
            if len(ctx_ids) + n_mid <= budget or (not pre and not suf):
                return ctx_ids, dropped
            if (dropped % 2 == 0 and pre) or not suf:
                pre.pop(0)
            else:
                suf.pop()
            dropped += 1

    def _nll(self, ctx_ids, mid_ids):
        torch = self.torch
        ids = torch.tensor([ctx_ids + mid_ids], device=self.dev)
        start = len(ctx_ids)
        with torch.no_grad():
            h = self.model.model(input_ids=ids, use_cache=False).last_hidden_state[0]
            logits = self.model.lm_head(h[start-1:start-1+len(mid_ids)]).float()
        tgt = ids[0, start:]
        nll = -torch.log_softmax(logits, dim=-1).gather(1, tgt.unsqueeze(1)).squeeze(1)
        return float(nll.mean())

    def entropy(self, prefix_lines, middle, suffix_lines):
        torch = self.torch
        mid_ids = (self.ids(middle) if middle else []) + self.eom
        mid_ids = mid_ids[:MAX_TOKENS - 64]
        budget = MAX_TOKENS
        while True:
            ctx_ids, dropped = self._fit(prefix_lines, suffix_lines, len(mid_ids), budget)
            try:
                e = self._nll(ctx_ids, mid_ids)
                return e, len(mid_ids), dropped, budget
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if budget <= 512:
                    raise
                budget -= 256
                print(f'    OOM -> retrying with {budget} tokens')


def load_prems(name, results_csv):
    df = pd.read_csv(f'{BASE}/{results_csv}')
    ks = df.killed.fillna(0) + df.survived.fillna(0)
    df['assessable'] = (df.status == 'OK') & (ks > 0)
    df['prmsc_num'] = pd.to_numeric(df.prmsc, errors='coerce')
    if name == 'd4c':
        pass
    elif name == 'wang':
        df = df.dropna(subset=['bug_id','tool'])
        df['slug'] = df['bug_id'].str.strip() + '_' + df['tool'].str.strip()
    elif name == 'prism':
        df['slug'] = df['bug_id'].str.strip() + '_' + df['tool'].str.strip()
    elif name == 'repairllama':
        df['slug'] = df['bug_id'].str.strip() + '_' + df['patch_file'].str.replace('.diff','',regex=False).str.strip()
    elif name == 'srepair':
        df = df.reset_index(drop=True)
        df['slug'] = df['bug_id'].str.strip() + '_' + df.index.astype(str)
    df['_seq'] = df.groupby('slug').cumcount()
    return df.set_index(['slug','_seq'])[['assessable','prmsc_num']]


def run(dataset_name, model_name, out_dir=BASE):
    patches_csv, results_csv = DATASETS[dataset_name]
    out_csv = f'{out_dir}/{dataset_name}_entropy_delta_results.csv'

    if dataset_name == 'd4c':
        import csv as _csv, sys, textwrap
        _csv.field_size_limit(sys.maxsize)
        with open(f'{BASE}/{D4C_PRED_CSV}', newline='', encoding='utf-8') as _f:
            _pred = {r['slug']: r for r in _csv.DictReader(_f)}
        _prems_df = pd.read_csv(f'{BASE}/{results_csv}')
        ks = _prems_df.killed.fillna(0) + _prems_df.survived.fillna(0)
        _prems_df['assessable'] = (_prems_df.status == 'OK') & (ks > 0)
        _prems_df['prmsc_num'] = pd.to_numeric(_prems_df.prmsc, errors='coerce')
        _prems_df['_seq'] = _prems_df.groupby('slug').cumcount()
        def _norm(c): return textwrap.dedent(str(c).replace('\r\n','\n').replace('\r','\n').strip('\n'))
        rows = []
        for _, r in _prems_df.iterrows():
            sl = r['slug']
            p = _pred.get(sl)
            bug_code = _norm(p['bug']) if p else ''
            patch_code = _norm(p['fix']) if p else ''
            rows.append({'dataset':'d4c','slug':sl,'label':int(r['label']),
                         'bug_code':bug_code,'patch_code':patch_code,
                         '_missing': p is None})
        patches = pd.DataFrame(rows)
        patches['_seq'] = patches.groupby('slug').cumcount()
    else:
        patches = pd.read_csv(f'{BASE}/{patches_csv}')
        patches['_seq'] = patches.groupby('slug').cumcount()
    prems   = load_prems(dataset_name, results_csv)

    done = set()
    if os.path.exists(out_csv):
        _d = pd.read_csv(out_csv); _d['_seq'] = _d.groupby('slug').cumcount()
        done = set(zip(_d.slug, _d._seq))
        print(f'Resuming {dataset_name}: {len(done)} already done')

    new_file = not os.path.exists(out_csv)
    out = open(out_csv, 'a', newline='')
    w = csv.DictWriter(out, fieldnames=FIELDS)
    if new_file:
        w.writeheader()

    scorer = None
    todo = patches[~patches.apply(lambda r: (r['slug'], r['_seq']) in done, axis=1)]

    for k, (_, r) in enumerate(todo.iterrows(), 1):
        slug = r['slug']; seq = int(r['_seq'])
        try:
            pm = prems.loc[(slug, seq)]; assessable, prmsc = bool(pm['assessable']), pm['prmsc_num']
        except KeyError:
            assessable, prmsc = False, ''
        row = {'dataset': dataset_name, 'slug': slug, 'label': int(r['label']),
               'assessable': assessable, 'prmsc': prmsc}

        if r.get('_missing'):
            row['status'] = 'NOT_IN_PRED'
        else:
            reg = patch_region(r['bug_code'], r['patch_code'])
        if not r.get('_missing') and reg is None:
            row['status'] = 'NO_CHANGE'
        elif not r.get('_missing') and reg is not None:
            if scorer is None:
                scorer = Scorer(model_name)
            pre, orig, new, suf = reg
            e_o, n_o, d_o, t_o = scorer.entropy(pre, join(orig), suf)
            e_p, n_p, d_p, t_p = scorer.entropy(pre, join(new), suf)
            row.update(e_orig=f'{e_o:.4f}', e_patch=f'{e_p:.4f}',
                       entropy_delta=f'{e_o - e_p:.4f}',
                       token_budget=min(t_o, t_p), status='OK')
            scorer.torch.cuda.empty_cache()

        w.writerow(row); out.flush()
        print(f"[{k}/{len(todo)}] {slug}: {row['status']} "
              f"{'dE='+row['entropy_delta'] if row['status']=='OK' else ''}")
    out.close()


def boot_auc(y, s, n=10000, seed=42):
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) > 1:
            vals.append(roc_auc_score(y[idx], s[idx]))
    if len(vals) < 100:
        return float('nan'), float('nan')
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

def evaluate(dataset_name, out_dir=BASE):
    from sklearn.metrics import roc_auc_score
    out_csv = f'{out_dir}/{dataset_name}_entropy_delta_results.csv'
    df = pd.read_csv(out_csv)
    ok = df[df.status == 'OK'].copy()
    ok['label'] = pd.to_numeric(ok['label'], errors='coerce')
    ok['overfit'] = 1 - ok['label']
    ok['prmsc'] = pd.to_numeric(ok['prmsc'], errors='coerce')
    ok['entropy_delta'] = pd.to_numeric(ok['entropy_delta'], errors='coerce')

    print(f"\n{'='*60}")
    print(f"{dataset_name.upper()}: {len(df)} total, {len(ok)} scored OK")

    def report(label, d):
        n_c = int((d.label==1).sum()); n_o = int((d.label==0).sum())
        print(f"\n  {label}: n={len(d)} (correct={n_c}, overfitting={n_o})")
        if n_c == 0 or n_o == 0:
            print("    only one class -- AUC undefined"); return
        for col, lbl in [('prmsc','PReMS'), ('entropy_delta','Entropy-Delta')]:
            sub = d.dropna(subset=[col])
            if sub.empty: continue
            auc = roc_auc_score(sub.overfit, -sub[col])
            lo, hi = boot_auc(sub.overfit, -sub[col])
            print(f"    {lbl:14s} AUC={auc:.3f} [{lo:.3f}, {hi:.3f}]")

    both = ok[ok.assessable.astype(bool) == True]
    report('Same patches as PReMS (assessable)', both)
    report('All patches Entropy-Delta could score', ok)

## Sanity check — PReMS AUCs should match your known values

In [ ]:
import pandas as pd; from sklearn.metrics import roc_auc_score

def add_seq(df): df=df.copy(); df['_seq']=df.groupby('slug').cumcount(); return df

_d4c = pd.read_csv(f'{CONTENT}/d4c_prms_results.csv')
_ks = _d4c.killed.fillna(0)+_d4c.survived.fillna(0)
_d4c['assessable'] = (_d4c.status=='OK')&(_ks>0)
_d4c['prmsc_num'] = pd.to_numeric(_d4c.prmsc, errors='coerce')
_a = _d4c[_d4c.assessable].dropna(subset=['prmsc_num'])
print(f'd4c: assessable={len(_a)}, PReMS AUC={round(roc_auc_score(1-_a.label, -_a.prmsc_num),3)}')

configs = {
    'wang':        ('wang_prms_results.csv',        lambda d: d.dropna(subset=['bug_id','tool']).assign(slug=lambda x: x.bug_id.str.strip()+'_'+x.tool.str.strip())),
    'prism':       ('prism_prms_results.csv',       lambda d: d.assign(slug=lambda x: x.bug_id.str.strip()+'_'+x.tool.str.strip())),
    'repairllama': ('repairllama_prms_results.csv', lambda d: d.assign(slug=lambda x: x.bug_id.str.strip()+'_'+x.patch_file.str.replace('.diff','',regex=False).str.strip())),
    'srepair':     ('srepair_prms_results.csv',     lambda d: d.reset_index(drop=True).assign(slug=lambda x: x.bug_id.str.strip()+'_'+x.index.astype(str))),
}
exp_files = {n: f'{n}_patches_export.csv' for n in configs}

for name, (rcsv, slug_fn) in configs.items():
    r = slug_fn(pd.read_csv(f'{CONTENT}/{rcsv}'))
    r['prmsc_num'] = pd.to_numeric(r.prmsc, errors='coerce')
    ks = r.killed.fillna(0)+r.survived.fillna(0)
    r['assessable'] = (r.status=='OK')&(ks>0)
    r = add_seq(r)
    e = add_seq(pd.read_csv(f'{CONTENT}/{exp_files[name]}'))
    m = e.merge(r[['slug','_seq','assessable','prmsc_num']], on=['slug','_seq'], how='left')
    a = m[m.assessable==True].dropna(subset=['prmsc_num'])
    if len(a['label'].unique()) < 2: print(f'{name}: only one class'); continue
    auc = round(roc_auc_score(1-a.label, -a.prmsc_num), 3)
    print(f'{name}: assessable={len(a)}, PReMS AUC={auc}')

d4c: assessable=52, PReMS AUC=0.724
wang: assessable=243, PReMS AUC=0.768
prism: assessable=658, PReMS AUC=0.772
repairllama: assessable=127, PReMS AUC=0.683
srepair: assessable=74, PReMS AUC=0.639


## Run each dataset
Each cell is independent and resumable. Results go to `OUT_DIR`.

In [ ]:
run('d4c', 'facebook/incoder-6B', OUT_DIR)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

[1/99] Math_41: OK dE=0.8186
[2/99] Time_18: OK dE=-0.0067
[3/99] Math_58: OK dE=1.0851
[4/99] Lang_11: OK dE=0.4256
[5/99] Lang_44: OK dE=0.3886
[6/99] Closure_119: OK dE=9.2550
[7/99] Time_16: OK dE=0.1722
[8/99] Chart_13: OK dE=0.0006
[9/99] Chart_5: OK dE=0.4761
[10/99] Closure_58: OK dE=-1.0191
[11/99] Closure_161: OK dE=0.6132
[12/99] Time_9: OK dE=-0.0411
[13/99] Closure_94: OK dE=0.2694
[14/99] Chart_1: OK dE=-0.5936
[15/99] Chart_6: OK dE=1.7959
[16/99] Chart_9: OK dE=-0.3160
[17/99] Chart_11: OK dE=0.7633
[18/99] Chart_24: OK dE=0.3047
[19/99] Closure_2: OK dE=0.5030
[20/99] Closure_5: OK dE=0.0352
[21/99] Closure_36: OK dE=11.1930
[22/99] Closure_38: OK dE=-0.3154
[23/99] Closure_44: OK dE=0.5291
[24/99] Closure_52: OK dE=10.0667
[25/99] Closure_56: OK dE=0.7629
[26/99] Closure_57: OK dE=1.6759
[27/99] Closure_62: OK dE=-0.0449
[28/99] Closure_64: OK dE=0.7488
[29/99] Closure_65: OK dE=0.3282
[30/99] Closure_66: OK dE=0.3149
[31/99] Closure_73: OK dE=0.9846
[32/99] Closure_7

In [ ]:
evaluate('d4c', OUT_DIR)


D4C: 99 total, 99 scored OK

  Same patches as PReMS (assessable): n=52 (correct=44, overfitting=8)
    PReMS          AUC=0.724 [0.536, 0.892]
    Entropy-Delta  AUC=0.642 [0.455, 0.817]

  All patches Entropy-Delta could score: n=99 (correct=86, overfitting=13)
    PReMS          AUC=0.724 [0.536, 0.892]
    Entropy-Delta  AUC=0.546 [0.394, 0.695]


In [ ]:
run('wang', 'facebook/incoder-6B', OUT_DIR)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

[1/421] Math-58_Arja: OK dE=0.5391
[2/421] Math-98_Arja: OK dE=1.1880
[3/421] Math-4_AVATAR: OK dE=0.2271
[4/421] Math-59_AVATAR: OK dE=0.2145
[5/421] Math-89_AVATAR: OK dE=1.2124
[6/421] Lang-6_AVATAR: OK dE=-0.0071
[7/421] Lang-7_AVATAR: OK dE=0.1001
[8/421] Math-70_Cardumen: OK dE=0.5528
[9/421] Math-22_FixMiner: OK dE=-0.5906
[10/421] Math-30_FixMiner: OK dE=0.0958
[11/421] Math-34_FixMiner: OK dE=0.3598
[12/421] Math-57_FixMiner: OK dE=1.6854
[13/421] Math-70_FixMiner: OK dE=0.5528
[14/421] Math-75_FixMiner: OK dE=0.6332
[15/421] Math-79_FixMiner: OK dE=-0.3710
[16/421] Math-70_jGenProg: OK dE=0.5528
[17/421] Math-82_jMutRepair: OK dE=0.1820
[18/421] Math-85_jMutRepair: OK dE=0.2947
[19/421] Math-58_kPAR: OK dE=0.5391
[20/421] Math-70_kPAR: OK dE=0.5528
[21/421] Math-75_kPAR: OK dE=0.6332
[22/421] Math-89_kPAR: OK dE=1.2124
[23/421] Math-41_SimFix: OK dE=-0.9766
[24/421] Math-5_SimFix: OK dE=-0.6819
[25/421] Math-53_SimFix: OK dE=10.0863
[26/421] Math-57_SimFix: OK dE=1.6854
[27/4

In [ ]:
evaluate('wang', OUT_DIR)


WANG: 421 total, 421 scored OK

  Same patches as PReMS (assessable): n=243 (correct=42, overfitting=201)
    PReMS          AUC=0.768 [0.705, 0.825]
    Entropy-Delta  AUC=0.700 [0.616, 0.779]

  All patches Entropy-Delta could score: n=421 (correct=77, overfitting=344)
    PReMS          AUC=0.601 [0.524, 0.679]
    Entropy-Delta  AUC=0.671 [0.605, 0.732]


In [ ]:
run('prism', 'facebook/incoder-6B', OUT_DIR)

Resuming prism: 788 already done


In [ ]:
evaluate('prism', OUT_DIR)


PRISM: 788 total, 788 scored OK

  Same patches as PReMS (assessable): n=658 (correct=107, overfitting=551)
    PReMS          AUC=0.772 [0.721, 0.821]
    Entropy-Delta  AUC=0.693 [0.646, 0.737]

  All patches Entropy-Delta could score: n=788 (correct=137, overfitting=651)
    PReMS          AUC=0.772 [0.721, 0.821]
    Entropy-Delta  AUC=0.693 [0.651, 0.733]


In [ ]:
run('repairllama', 'facebook/incoder-6B', OUT_DIR)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

[1/160] Chart_11_Chart_11_repairllama-ir4_patch_0: OK dE=0.3893
[2/160] Chart_11_Chart_11_repairllama-ir4_patch_7: OK dE=-0.1087
[3/160] Chart_11_Chart_11_repairllama-ir4_patch_8: OK dE=-0.5502
[4/160] Chart_12_Chart_12_repairllama-ir4_patch_0: OK dE=-2.1098
[5/160] Chart_1_Chart_1_repairllama-ir4_patch_0: OK dE=0.6503
[6/160] Chart_20_Chart_20_repairllama-ir4_patch_0: OK dE=0.5018
[7/160] Chart_26_Chart_26_repairllama-ir4_patch_0: OK dE=0.1557
[8/160] Chart_26_Chart_26_repairllama-ir4_patch_2: OK dE=0.1554
[9/160] Chart_26_Chart_26_repairllama-ir4_patch_3: OK dE=0.1487
[10/160] Chart_26_Chart_26_repairllama-ir4_patch_4: OK dE=0.1561
[11/160] Chart_26_Chart_26_repairllama-ir4_patch_5: OK dE=0.1548
[12/160] Chart_26_Chart_26_repairllama-ir4_patch_9: OK dE=0.1574
[13/160] Chart_4_Chart_4_repairllama-ir4_patch_0: OK dE=0.0937
[14/160] Chart_4_Chart_4_repairllama-ir4_patch_8: OK dE=0.0924
[15/160] Chart_4_Chart_4_repairllama-ir4_patch_9: OK dE=0.0888
[16/160] Chart_7_Chart_7_repairllama-ir

In [ ]:
evaluate('repairllama', OUT_DIR)


REPAIRLLAMA: 160 total, 160 scored OK

  Same patches as PReMS (assessable): n=127 (correct=54, overfitting=73)
    PReMS          AUC=0.683 [0.593, 0.767]
    Entropy-Delta  AUC=0.423 [0.324, 0.527]

  All patches Entropy-Delta could score: n=160 (correct=63, overfitting=97)
    PReMS          AUC=0.683 [0.593, 0.767]
    Entropy-Delta  AUC=0.428 [0.339, 0.521]


In [ ]:
run('srepair', 'facebook/incoder-6B', OUT_DIR)

Resuming srepair: 158 already done


In [ ]:
evaluate('srepair', OUT_DIR)


SREPAIR: 158 total, 158 scored OK

  Same patches as PReMS (assessable): n=74 (correct=18, overfitting=56)
    PReMS          AUC=0.639 [0.477, 0.791]
    Entropy-Delta  AUC=0.427 [0.271, 0.585]

  All patches Entropy-Delta could score: n=158 (correct=33, overfitting=125)
    PReMS          AUC=0.575 [0.464, 0.687]
    Entropy-Delta  AUC=0.436 [0.329, 0.548]


## Summary — all datasets

In [ ]:
for ds in ['d4c','wang','prism','repairllama','srepair']:
    try:
        evaluate(ds, OUT_DIR)
    except FileNotFoundError:
        print(f'{ds}: not yet run')


D4C: 99 total, 99 scored OK

  Same patches as PReMS (assessable): n=52 (correct=44, overfitting=8)
    PReMS          AUC=0.724 [0.536, 0.892]
    Entropy-Delta  AUC=0.642 [0.455, 0.817]

  All patches Entropy-Delta could score: n=99 (correct=86, overfitting=13)
    PReMS          AUC=0.724 [0.536, 0.892]
    Entropy-Delta  AUC=0.546 [0.394, 0.695]

WANG: 421 total, 421 scored OK

  Same patches as PReMS (assessable): n=243 (correct=42, overfitting=201)
    PReMS          AUC=0.768 [0.705, 0.825]
    Entropy-Delta  AUC=0.700 [0.616, 0.779]

  All patches Entropy-Delta could score: n=421 (correct=77, overfitting=344)
    PReMS          AUC=0.601 [0.524, 0.679]
    Entropy-Delta  AUC=0.671 [0.605, 0.732]

PRISM: 788 total, 788 scored OK

  Same patches as PReMS (assessable): n=658 (correct=107, overfitting=551)
    PReMS          AUC=0.772 [0.721, 0.821]
    Entropy-Delta  AUC=0.693 [0.646, 0.737]

  All patches Entropy-Delta could score: n=788 (correct=137, overfitting=651)
    PReMS 